# Prompt 2 — Shortcut probe
Fail-closed, deterministic Stage 0.5 runner. Run All succeeds only when exact internal-epoch 0/25/125 snapshots are present; it never creates or rewrites legacy checkpoints.

In [ ]:
from pathlib import Path

# Single editable configuration cell.
DRIVE_ROOT = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
DATASET_ROOT = DRIVE_ROOT / 'dataset_cache_colab_v1/Dataset801_IAC_LR'
TRACKB_CACHE_ROOT = DRIVE_ROOT / 'sdf_cache_backup'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
SPLITS_PATH = DRIVE_ROOT / 'configs_cache/splits.json'
RUNS_ROOT = DRIVE_ROOT
RUN_DIR = DRIVE_ROOT / 'flow_fold0'
CHECKPOINT_ROOT = RUN_DIR / 'prompt2_exact_checkpoints'
CHECKPOINTS = {0: CHECKPOINT_ROOT/'epoch_000.pt', 25: CHECKPOINT_ROOT/'epoch_025.pt', 125: CHECKPOINT_ROOT/'epoch_125.pt'}
IDENTITY_BASELINE = OUTPUT_ROOT / 'baselines/identity_prior.json'
ANALYSIS_OUTPUT = OUTPUT_ROOT / 'analysis'
WORK_DIR = OUTPUT_ROOT / 'prompt2/shortcut_probe_work'
LOCAL_TEMP_PARENT = Path('/content/prompt2_staging')
REPO_URL = 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git'
PINNED_COMMIT = 'b1c294def65d7625e44a9e4d9be280033bd31dcc'
REPO = Path('/content/ToothFairy3-IAC-Segmentation-Flow')
DEVICE = 'cuda'
CASES = 12
PATCHES_PER_STRATUM = 2
THICKENING_CASES = 5
BATCH_SIZE = 6
BOOTSTRAP_ITERATIONS = 2000

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
if not REPO.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == PINNED_COMMIT, (head, PINNED_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements.txt')], check=True)
import torch
assert DEVICE == 'cuda' and torch.cuda.is_available(), 'Select a Colab GPU runtime'
print({'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
for name, path in [('DRIVE_ROOT', DRIVE_ROOT), ('DATASET_ROOT', DATASET_ROOT), ('TRACKB_CACHE_ROOT', TRACKB_CACHE_ROOT), ('SPLITS_PATH', SPLITS_PATH), ('RUN_DIR', RUN_DIR), ('IDENTITY_BASELINE', IDENTITY_BASELINE)]:
    assert path.exists(), f'{name} missing: {path}'
LOCAL_TEMP_PARENT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Exact internal epoch values are mandatory. Missing epoch-125 must stop here.
readiness_cmd = [sys.executable, 'analysis/shortcut_probe.py',
    '--identity-baseline', str(IDENTITY_BASELINE), '--runs-root', str(RUNS_ROOT),
    '--run-dir', str(RUN_DIR), '--readiness-only']
for epoch, path in CHECKPOINTS.items():
    readiness_cmd += ['--checkpoint', f'{epoch}={path}']
subprocess.run(readiness_cmd, cwd=REPO, check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/'], cwd=REPO, check=True)

In [ ]:
probe_cmd = [sys.executable, 'analysis/shortcut_probe.py',
    '--identity-baseline', str(IDENTITY_BASELINE), '--runs-root', str(RUNS_ROOT),
    '--run-dir', str(RUN_DIR), '--splits', str(SPLITS_PATH),
    '--images', str(DATASET_ROOT/'imagesTr'), '--labels', str(DATASET_ROOT/'labelsTr'),
    '--gt-sdf', str(TRACKB_CACHE_ROOT/'gt_sdf'),
    '--coarse-sdf', str(TRACKB_CACHE_ROOT/'coarse_sdf'),
    '--output-dir', str(ANALYSIS_OUTPUT), '--work-dir', str(WORK_DIR),
    '--local-temp-parent', str(LOCAL_TEMP_PARENT), '--device', DEVICE,
    '--cases', str(CASES), '--patches-per-stratum', str(PATCHES_PER_STRATUM),
    '--thickening-cases', str(THICKENING_CASES), '--batch-size', str(BATCH_SIZE),
    '--bootstrap-iterations', str(BOOTSTRAP_ITERATIONS)]
for epoch, path in CHECKPOINTS.items():
    probe_cmd += ['--checkpoint', f'{epoch}={path}']
subprocess.run(probe_cmd, cwd=REPO, check=True)

In [ ]:
import csv, hashlib, json
expected = ['shortcut_probe.csv', 'shortcut_probe_summary.json', 'fig1_shortcut.pdf', 'thickening_probe.csv', 'shortcut_probe_manifest.json']
for name in expected:
    path = ANALYSIS_OUTPUT/name
    assert path.is_file() and path.stat().st_size > 0, path
manifest = json.loads((ANALYSIS_OUTPUT/'shortcut_probe_manifest.json').read_text())
summary = json.loads((ANALYSIS_OUTPUT/'shortcut_probe_summary.json').read_text())
for name, expected_sha in manifest['artifacts'].items():
    actual = hashlib.sha256((ANALYSIS_OUTPUT/name).read_bytes()).hexdigest()
    assert actual == expected_sha, (name, actual, expected_sha)
with (ANALYSIS_OUTPUT/'shortcut_probe.csv').open() as handle:
    rows = list(csv.DictReader(handle))
assert rows and {int(row['checkpoint_epoch']) for row in rows} == {0, 25, 125}
assert {row['stratum'] for row in rows} == {'foreground', 'pure_background'}
assert (ANALYSIS_OUTPUT/'fig1_shortcut.pdf').read_bytes().startswith(b'%PDF')
print('Output validation PASS:', ANALYSIS_OUTPUT)

In [ ]:
print('Prompt 2 complete')
print('Git:', manifest['git']['head'])
print('Checkpoints:', {epoch: item['sha256'] for epoch, item in manifest['checkpoints'].items()})
print('Rows:', summary['counts'])
print(summary['claim_limit'])
print('Artifacts:', *[str(ANALYSIS_OUTPUT/name) for name in expected], sep='\n- ')